Stylistique numérique

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabays/32M7131/blob/master/Cours_08/Cours08.ipynb)

# Entraîner un classifieur avec _Spacy_

Simon Gabay

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licence Creative Commons" style="float: right" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a>

--------------------------------------------------

🚨🚨 **Note: Sélectionner `Edit`>`Notebook settings`. Sélectionner `GPU` pour `Hardware accelerator`**. Vous allez ainsi utiliser les GPU de google, ce que vous ne pouvez faire que de manière limitée sans abonnement.

--------------------------------------------------

J'installe [Spacy](https://spacy.io/), une bibliothèque python pour le TAL très répandue et très bien documentée.
L'installation se fait avec `pip`, un gestionnaire de paquets utilisé pour installer et gérer des paquets écrits en Python. Pour l'utiliser, je dois faire la commande `pip install` + nom de la bibliothèque que je veux utiliser.
L'utilisation de `pip` n'est pas du python mais une [commande shell](https://doc.ubuntu-fr.org/commande_shell), je dois commencer la ligne avec `!`. Attention à bien utiliser le paramètre `-U` pour utiliser la dernière version!

In [2]:
!pip install -U spacy

     |████████████████████████████████| 6.2 MB 8.7 MB/s 
     |████████████████████████████████| 10.1 MB 56.7 MB/s 
     |████████████████████████████████| 181 kB 75.5 MB/s 
     |████████████████████████████████| 42 kB 1.8 MB/s 
     |████████████████████████████████| 457 kB 72.5 MB/s 
     |████████████████████████████████| 653 kB 66.2 MB/s 
     |████████████████████████████████| 58 kB 7.6 MB/s 
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.2.0
    Uninstalling typing-extensions-4.2.0:
      Successfully uninstalled typing-extensions-4.2.0
  Attempting uninstall: catalogue
    Found existing installation: catalogue 1.0.0
    Uninstalling catalogue-1.0.0:
      Successfully uninstalled catalogue-1.0.0
  Attempting uninstall: srsly
    Found existing installation: srsly 1.0.5
    Uninstalling srsly-1.0.5:
      Successfully uninstalled srsly-1.0.5
  Attempting uninstall: smart-open
    Found existing installation: smart-open 6.0.0
    U

\Je vais maintenant pouvoir utiliser _Spacy_. Comme nous voulons utiliser un modèle de langue, je peux regarder [ce qui est proposé](https://spacy.io/models). Utilisons [le modèle multilingue](https://spacy.io/models/xx/) pour l'analyse d'entités nommées. Il est entraîné avec wikipedia. Il existe des modèles pour différentes langues, voire différents états de langue, mais l'utilisation de ce modèle rend ce tutoriel utilisable par des spécialistes de différentes langues.

In [3]:
!python -m spacy download xx_ent_wiki_sm

     |████████████████████████████████| 11.2 MB 8.0 MB/s 
✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')


Maintenant je dois charger des données d'entraînement. Un petit document annoté est disponible en ligne sur le dépôt GitHub du cours:
1. Créons un dossier où le mettre avec la commande `mkdir` (_make directory_)
2. Téléchargeons le document avec `wget`, un programme en ligne de commande non interactif de téléchargement de fichiers depuis le Web, et l'URL du fichier que l'on veut récupérer (au format brut)

In [4]:
!mkdir training_data
!wget https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_AGAMEMNON.tsv
!mv /content/BOYER_AGAMEMNON.tsv /content/training_data

--2022-05-16 16:59:31--  https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_AGAMEMNON.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 116078 (113K) [text/plain]
Saving to: ‘BOYER_AGAMEMNON.tsv’

BOYER_AGAMEMNON.tsv 100%[===================>] 113.36K  --.-KB/s    in 0.01s   

2022-05-16 16:59:31 (9.96 MB/s) - ‘BOYER_AGAMEMNON.tsv’ saved [116078/116078]



Je peux maintenant regarder à quoi ressemble ce que j'ai téléchargé. Pour cela je vais avoir besoin de transformer le fichier `.tsv` en _dataframe_ et en afficher les premières lignes. La bibliothèque python qui permet de manipuler les _dataframes_, que je dois donc télécharger avec pip

In [5]:
!pip install pandas
# j'importe pandas sous le nom "pd" (comme cela c'est plus court, donc plus simple)
import pandas as pd

Je peux maintenant transformer mes données en _dataframe_ et en afficher les dix premières lignes

In [6]:
df = pd.read_csv('/content/training_data/BOYER_AGAMEMNON.tsv', sep="\t", header=None)
df.head(17)

,0,1
0,Oui,O
1,",",O
2,Pylade,O
3,",",O
4,il,O
5,est,O
6,vrai,O
7,",",O
8,la,O
9,valeur,O


Je dois désormais importer un second texte, qui va me servir pour le jeu de _dev_ ou d'évaluation.

In [7]:
!mkdir training_data
!wget https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_AMOURSJUPITERSEMELE.tsv
!mv /content/BOYER_AMOURSJUPITERSEMELE.tsv /content/training_data

mkdir: cannot create directory ‘training_data’: File exists
--2022-05-16 16:59:35--  https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_AMOURSJUPITERSEMELE.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 141839 (139K) [text/plain]
Saving to: ‘BOYER_AMOURSJUPITERSEMELE.tsv’

BOYER_AMOURSJUPITER 100%[===================>] 138.51K  --.-KB/s    in 0.01s   

2022-05-16 16:59:35 (9.48 MB/s) - ‘BOYER_AMOURSJUPITERSEMELE.tsv’ saved [141839/141839]



Le fichier `.tsv` est impropre à être manipulé par _Spacy_ nous allons donc d'abord le transformer en `.json`. J'utilise une série de paramètres lors de cette conversion, notamment:
* `python -m spacy convert` pour la conversion
* Le fichier a traiter
* Le dossier ou stocker le résultat de la conversion
* `-t json` pour le format de sortie
* `-n 1` pour faire un document par phrae
* `-c iob` qui détecte que j'utilise une annotation qui suit le format standard _IOB_ (_inside, beginning, Outside_).

In [8]:
!mkdir training_json
!python -m spacy convert /content/training_data/BOYER_AGAMEMNON.tsv /content/training_json -t json -n 1 -c iob
!python -m spacy convert /content/training_data/BOYER_AMOURSJUPITERSEMELE.tsv /content/training_json -t json -n 1 -c iob -b xx_ent_wiki_sm

ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1 documents):
/content/training_json/BOYER_AGAMEMNON.json
ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1 documents):
/content/training_json/BOYER_AMOURSJUPITERSEMELE.json


_Spacy_ nous indique qu'il recommande de regrouper les phrases par 10: suivons sa recommendation

In [9]:
!python -m spacy convert /content/training_data/BOYER_AGAMEMNON.tsv /content/training_json -t json -n 10 -c iob
!python -m spacy convert /content/training_data/BOYER_AMOURSJUPITERSEMELE.tsv /content/training_json -t json -n 10 -c iob

ℹ Auto-detected token-per-line NER format
ℹ Grouping every 10 sentences into a document.
✔ Generated output file (1 documents):
/content/training_json/BOYER_AGAMEMNON.json
ℹ Auto-detected token-per-line NER format
ℹ Grouping every 10 sentences into a document.
✔ Generated output file (1 documents):
/content/training_json/BOYER_AMOURSJUPITERSEMELE.json


Affichons un petit extrait du résultat pour observer le résulat:

In [10]:
f = open('/content/training_json/BOYER_AGAMEMNON.json')
content = f.readlines()
print(*content[74:93])

                "orth":"valeur",
                 "space":" ",
                 "tag":"-",
                 "ner":"O"
               },
               {
                 "id":10,
                 "orth":"et",
                 "space":" ",
                 "tag":"-",
                 "ner":"O"
               },
               {
                 "id":11,
                 "orth":"l'",
                 "space":" ",
                 "tag":"-",
                 "ner":"O"
               },



Après cette étape intermédiaire, je n'ai plus qu'à convertir mon fichier `.json` dans le format spécifique de _Spacy_.

In [11]:
!mkdir training_spacy
!python -m spacy convert /content/training_json/BOYER_AGAMEMNON.json /content/training_spacy -t spacy
!python -m spacy convert /content/training_json/BOYER_AMOURSJUPITERSEMELE.json /content/training_spacy -t spacy

✔ Generated output file (101 documents):
/content/training_spacy/BOYER_AGAMEMNON.spacy
✔ Generated output file (108 documents):
/content/training_spacy/BOYER_AMOURSJUPITERSEMELE.spacy


## On prépare l'installation

Nous devons d'abord installer la bibliothèque _NVIDIA 9.2 cuda_. `dpkg` permet d'installer le paquet (qui se retrouve dans `/var`) que nous téléchargeons avec `wget`. `apt-get install` permet d'installer les dépendances du paquet que nous venons d'installer. [link text](https://)Pour contrôler que le compileur cuda est bien installé, on peut exécuter la commande `!nvcc --version` (nvcc pour _NVIDIA CUDA Compiler_).

In [12]:
!wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604–9–2-local_9.2.88–1_amd64.deb
!dpkg -i cuda-repo-ubuntu1604–9–2-local_9.2.88–1_amd64.deb
!apt-key add /var/cuda-repo-9–2-local/7fa2af80.pub
!apt-get update
!apt-get install cuda-9.2
!nvcc --version

--2022-05-16 17:00:11--  https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64
Resolving developer.nvidia.com (developer.nvidia.com)... 152.195.19.142
Connecting to developer.nvidia.com (developer.nvidia.com)|152.195.19.142|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://developer.nvidia.com/compute/cuda/9.2/prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 [following]
--2022-05-16 17:00:11--  https://developer.nvidia.com/compute/cuda/9.2/prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64
Reusing existing connection to developer.nvidia.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://developer.download.nvidia.com/compute/cuda/9.2/secure/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb?IT_9-2CHLoLz-nRPh7BmufOOcC1eNnxnqp-ee-NrImc5524doqs159AMWnkE26rZ1LzOCCFGOPWGmVinLUk40L4rtEgGgmuzOJpAoMRW1BKaP

Il faut désormais installer la bibliothèque la célèbre bibliothèque de _machine learning_ développée par Facebook nommée _PyTorch_. Il faut faire attention à bien installer la même version que précedemment (ici `9.2`).

In [13]:
!pip install torch==1.7.1+cu92 torchvision==0.8.2+cu92 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html

Looking in links: https://download.pytorch.org/whl/torch_stable.html
     |████████████████████████████████| 577.3 MB 3.5 kB/s 
     |████████████████████████████████| 12.5 MB 740 kB/s 
     |████████████████████████████████| 7.6 MB 6.9 MB/s 
  Attempting uninstall: torch
    Found existing installation: torch 1.11.0+cu113
    Uninstalling torch-1.11.0+cu113:
      Successfully uninstalled torch-1.11.0+cu113
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.12.0+cu113
    Uninstalling torchvision-0.12.0+cu113:
      Successfully uninstalled torchvision-0.12.0+cu113
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 0.11.0+cu113
    Uninstalling torchaudio-0.11.0+cu113:
      Successfully uninstalled torchaudio-0.11.0+cu113
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtext 0.12.0 requires torch

Il faut désormais installer les transformeurs _Spacy_ et modiier le _path_ (qu'il utilise le cuda que nous venons d'installer!).

In [14]:
!pip install -U spacy[cuda92,transformers]
!export CUDA_PATH=”/usr/local/cuda-9.2"
!export LD_LIBRARY_PATH=$CUDA_PATH/lib64:$LD_LIBRARY_PATH

     |████████████████████████████████| 55.0 MB 60 kB/s 
     |████████████████████████████████| 51 kB 152 kB/s 
     |████████████████████████████████| 3.8 MB 35.1 MB/s 
     |████████████████████████████████| 1.1 MB 57.2 MB/s 
     |████████████████████████████████| 880 kB 55.8 MB/s 
     |████████████████████████████████| 84 kB 1.8 MB/s 
     |████████████████████████████████| 596 kB 56.4 MB/s 
     |████████████████████████████████| 6.6 MB 45.9 MB/s 
  Created wheel for sacremoses: filename=sacremoses-0.0.53-py3-none-any.whl size=895260 sha256=46e595dd685226949f4a296e82b94313f251e8e6e13912e1f118eec55f531ce7
  Stored in directory: /root/.cache/pip/wheels/87/39/dd/a83eeef36d0bf98e7a4d1933a4ad2d660295a40613079bafc9
Successfully built sacremoses
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13
/bin/bash: -c: line 0: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 1: syn

J'installe _cupy_, une bibliothèque pour manipuler des matrices optimisée pour les GPU (comme _numpy_ pour les CPU).

In [15]:
!pip install cupy==9.6.0

     |████████████████████████████████| 1.6 MB 9.1 MB/s 
  Created wheel for cupy: filename=cupy-9.6.0-cp37-cp37m-linux_x86_64.whl size=53787943 sha256=c27e8955a5d8146877dbd7633db41e63340f276fe431b65a4a96b34f663cf7ff
  Stored in directory: /root/.cache/pip/wheels/57/44/0b/5c540f032d681b9c7bcafad447177f7e356cba004e48df1d2a
Successfully built cupy


Je dois désormais configurer mon entraînement. Pour cela je vais devoir préparer un fichier de configuration, qui contient tous les paramètres pour mon entraînement. Un [formulaire en ligne](https://spacy.io/usage/training#quickstart) permet de le faire simplement.

🚨 **Pensez à bien changer le chemin pour le _dev_ et le _train_!** Cf. [ici](https://github.com/gabays/32M7131/blob/c480c29386874adb1b1a1def471f34cd02489e13/Cours_08/Data/config.cfg#L5)

In [16]:
!wget https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/config.cfg

--2022-05-16 17:20:28--  https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/config.cfg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1793 (1.8K) [text/plain]
Saving to: ‘config.cfg’

config.cfg          100%[===================>]   1.75K  --.-KB/s    in 0s      

2022-05-16 17:20:28 (34.7 MB/s) - ‘config.cfg’ saved [1793/1793]



Il faut désormais compléter le fichier de configuration avec les paramètres nécessaires à BERT. Il peut être utile d'aller ajuster certaines valeurs dans celui-ci (_batch size_, _learning rate_…). Il est extrêmement difficile de connaître les bons paramètres à l'avance: il faut normalement relancer plusieurs fois l'entraînement avec différents paramètres et déduire des observations la meilleure configuration.

In [17]:
!python -m spacy init fill-config /content/config.cfg /content/config_spacy.cfg

✔ Auto-filled config with all values
✔ Saved config
/content/config_spacy.cfg
You can now add your data and train your pipeline:
python -m spacy train config_spacy.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


Afin de contrôler que notre fichier de configuration est bon, il est possible d'utiliser une fonction `debug`:

In [18]:
!python -m spacy debug data /content/config_spacy.cfg


============================ Data file validation ============================
Downloading: 100% 28.0/28.0 [00:00<00:00, 28.5kB/s]
Downloading: 100% 625/625 [00:00<00:00, 435kB/s]
Downloading: 100% 851k/851k [00:00<00:00, 2.32MB/s]
Downloading: 100% 1.64M/1.64M [00:00<00:00, 3.88MB/s]
Downloading: 100% 641M/641M [00:18<00:00, 37.3MB/s]
Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if yo

Je peux désormais lancer mon rentraînement! Pour analyser les résultats, on utilise traditionnellement:

* la _loss_, ou fonction de perte, est une fonction qui évalue l’écart entre les prédictions réalisées par le réseau de neurones et les valeurs réelles des observations utilisées pendant l’apprentissage. Plus le résultat de cette fonction est minimisé, plus le réseau de neurones est performant. Sa minimisation, c’est-à-dire réduire au minimum l’écart entre la valeur prédite et la valeur réelle pour une observation donnée, se fait en ajustant les différents poids du réseau de neurones.
* `ENTS_P`: la [précision](https://en.wikipedia.org/wiki/Precision_and_recall) (_precision_)
* `ENTS_R`: le [rappel](https://en.wikipedia.org/wiki/Precision_and_recall) (_recall_)
* `ENTS_F`: le [score F1](https://en.wikipedia.org/wiki/F-score) (_F-score_) qui fait la synthèse des deux mesures précédentes.

<img alt="Licence Creative Commons" style="float: left" height="500px" src="https://upload.wikimedia.org/wikipedia/commons/f/ff/Precisionrappel.svg" />


In [19]:
!python -m spacy train -g 0 /content/config_spacy.cfg --output ./

ℹ Saving to output directory: .
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
[2022-05-16 17:21:22,939] [INFO] Set up nlp object from config
[2022-05-16 17:21:22,949] [INFO] Pipeline: ['transformer', 'ner']
[2022-05-16 17:21:22,953] [INFO] Created vocabulary
[2022-05-16 17:21:22,954] [INFO] Finished initializing nlp object
Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining mo

Le meilleur modèle est sauvegardé dans le dossier `model-best`.

# Tester, utiliser

Nous pouvons désormais tester le résultat sur un troisième document. Téléchargeons-le, et transformons-le dans le format attendu:

In [21]:
!wget https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_ARISTODEME.tsv
!mv /content/BOYER_ARISTODEME.tsv /content/training_data
!python -m spacy convert /content/training_data/BOYER_ARISTODEME.tsv /content/training_json -t json -n 10 -c iob
!python -m spacy convert /content/training_json/BOYER_ARISTODEME.json /content/training_spacy -t spacy

--2022-05-16 18:51:32--  https://raw.githubusercontent.com/gabays/32M7131/master/Cours_08/Data/BOYER_ARISTODEME.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 94623 (92K) [text/plain]
Saving to: ‘BOYER_ARISTODEME.tsv’

BOYER_ARISTODEME.ts 100%[===================>]  92.41K  --.-KB/s    in 0.01s   

2022-05-16 18:51:32 (8.71 MB/s) - ‘BOYER_ARISTODEME.tsv’ saved [94623/94623]

ℹ Auto-detected token-per-line NER format
ℹ Grouping every 10 sentences into a document.
✔ Generated output file (1 documents):
/content/training_json/BOYER_ARISTODEME.json
✔ Generated output file (75 documents):
/content/training_spacy/BOYER_ARISTODEME.spacy


Maintenant, testons le modèle avec ce fichier test

In [23]:
!mkdir results
!python -m spacy evaluate /content/model-best /content/training_spacy/BOYER_ARISTODEME.spacy  -dp /content/results

mkdir: cannot create directory ‘results’: File exists
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

================================== Results ==================================

TOK     100.00
NER P   0.00  
NER R   0.00  
NER F   0.00  
SPEED   202   


=============================== NER (per type) ===============================

         P      R      F
LOC   0.00   0.00   0.00
C-B   0.00   0.00   0.00
C-I   0.00   0.00   0.00

/usr/local/lib/python3.7/dist-packages/spacy/displacy/__init__.py:205: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)
✔ Generated 25 parses as HTML
/content/results


On peut télécharger le résultat dans une fichier HTML facilement lisible

In [25]:
from google.colab import files
files.download('/content/results/entities.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Il est possible de faire un test sur une phrase inventée:

In [34]:
#J'importe spacy pour l'utiliser ligne 7
import spacy
#J'invente un phrase une phrase (je laisse volontairement
#une coquille et simuler une erreur d'OCR)
test = "je vais en Asie, à Sparte et en Messéuie"
#Je charge mon modèle
nlp = spacy.load('/content/model-best')
#J'applique mon modèle à ma phrase de test
doc = nlp(test)
#Je demande à Spacy de m'identifier les lieux
for ent in doc.ents:
  print(ent.text)

Asie
Sparte
Messéuie
